In [1]:
from genericpath import exists
import pdb
from pickletools import uint8
from turtle import pd
import torchvision
import torch
import os
from pytorch_grad_cam import GradCAM, ScoreCAM, GradCAMPlusPlus, AblationCAM, XGradCAM, EigenCAM, FullGrad
# from torchknickknacks import modelutils
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms
from pytorch_grad_cam.utils.image import show_cam_on_image

import numpy as np
import cv2
import requests
from pytorch_grad_cam.ablation_layer import AblationLayerFasterRCNN
from pytorch_grad_cam.utils.model_targets import FasterRCNNBoxScoreTarget
from pytorch_grad_cam.utils.reshape_transforms import fasterrcnn_reshape_transform
from pytorch_grad_cam.ablation_layer import AblationLayerFasterRCNN
import random
import glob
from skimage.measure import label, regionprops, regionprops_table
from skimage import data, filters, measure, morphology
import pandas as pd
import wandb
from tqdm import tqdm
from skimage import data
from skimage.color import rgb2hed, hed2rgb
import sys
sys.path.insert(0, '../')
from models.model_mrcnn import _default_mrcnn_config, build_default


In [2]:
input_path ="/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Train_val_LB/val/"

In [4]:
# model_input_path = '/home/mahirwar/Desktop/Monika/npsad_data/monika/LBD/models/mrcnn_models/royal-butterfly-145_mrcnn_model_10.pth'
model_input_path =  '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/models/mrcnn_models/trim-puddle-222_mrcnn_model_50.pth'

In [5]:
ground_truth = pd.read_csv("/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Reports/image_level_ground_truths.csv")

In [6]:
test_config = dict(
        batch_size = 1,
        num_classes = 2
    )

model_config = _default_mrcnn_config(num_classes=1 + test_config['num_classes']).config
model = build_default(model_config, im_size=1024)

/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/torchvision/models/_utils.py:135: UserWarning: Using 'backbone_name' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(
/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet152_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet152_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [11]:
class ExplainPredictions():
    
    # TODO fix the visualization flags
    def __init__(self, model, model_input_path, test_input_path, detection_threshold, wandb, save_result, ablation_cam, save_thresholds, ground_truth):
        self.model = model
        self.model_input_path = model_input_path
        self.test_input_path = test_input_path
        self.detection_threshold = detection_threshold
        self.wandb = wandb
        self.save_result = save_result
        self.ablation_cam = ablation_cam
        self.save_thresholds = save_thresholds
        self.class_names = ['True','Pre']
        self.class_to_colors = {'True': (255, 0, 0),'Pre':(0,255,0)}
        self.result_save_dir = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Reports/figures/" + model_input_path.split("/")[-1]
        
        self.colors = np.random.uniform(0, 255, size=(len(self.class_names), 2))
        print(self.colors)
        self.column_names = ["image_name", "region", "region_mask", "label", 
                            "confidence", "brown_pixels", "centroid", 
                            "eccentricity", "area", "equivalent_diameter"]
        self.results_path = ""
        self.masks_path = "" 
        self.detections_path = "" 
        self.ablations_path = ""
        self.quantify_path = ""
        self.ground_truth=ground_truth
      
    def get_brown_pixel_cnt(self, img, img_name):

        # Separate the stains from the IHC image
        ihc_hed = rgb2hed(img)

        # Create an RGB image for each of the stains
        null = np.zeros_like(ihc_hed[:, :, 0])
        ihc_h = hed2rgb(np.stack((ihc_hed[:, :, 0], null, null), axis=-1))
        ihc_e = hed2rgb(np.stack((null, ihc_hed[:, :, 1], null), axis=-1))
        ihc_d = hed2rgb(np.stack((null, null, ihc_hed[:, :, 2]), axis=-1))
        ihc_d = ihc_d.astype('float32')

        gray = cv2.cvtColor(ihc_d, cv2.COLOR_RGB2GRAY)
        gray = cv2.GaussianBlur(gray, (5, 5), 0)
        x = gray[gray<0.35]


        if len(x) != 0:
            if self.save_thresholds:
                # Display
                fig, axes = plt.subplots(2, 2, figsize=(7, 6), sharex=True, sharey=True)
                ax = axes.ravel()

                ax[0].imshow(img)
                ax[0].set_title("Original image")

                ax[1].imshow(ihc_h)
                ax[1].set_title("Hematoxylin")

                ax[2].imshow(gray)
                ax[2].set_title("gray")  # Note that there is no Eosin stain in this image

                ax[3].imshow(ihc_d)
                ax[3].set_title("DAB")

                for a in ax.ravel():
                    a.axis('off')

                fig.tight_layout()
            
                
                final_save_path = img_name + "_count_threhold.png"
                final_save_path = os.path.join(self.pixel_count_path, final_save_path)
                fig.savefig(final_save_path)
                plt.close()
            return len(x)
        return 0


    def get_outputs(self, input_tensor, model, threshold):
        with torch.no_grad():
            # forward pass of the image through the modle
            outputs = model(input_tensor)
        
        # get all the scores
        #print(outputs)
        scores = list(outputs[0]['scores'].detach().cpu().numpy())
        # print("\n scores", max(scores))
        # index of those scores which are above a certain threshold
        thresholded_preds_inidices = [scores.index(i) for i in scores if i > threshold]
        #print(thresholded_preds_inidices)
        thresholded_preds_count = len(thresholded_preds_inidices)
        #print(thresholded_preds_count)
        scores = scores[:thresholded_preds_count]
        # get the masks
        masks = (outputs[0]['masks']>0.5).squeeze().detach().cpu().numpy()
        # print("masks", masks)
        # discard masks for objects which are below threshold
        masks = masks[:thresholded_preds_count]
        # get the bounding boxes, in (x1, y1), (x2, y2) format
        boxes = [[(int(i[0]), int(i[1])), (int(i[2]), int(i[3]))]  for i in outputs[0]['boxes'].detach().cpu()]
        # discard bounding boxes below threshold value
        boxes = boxes[:thresholded_preds_count]
        # get the classes labels
        # print('labels', outputs[0]['labels'])
        #print(outputs[0]['labels'])
        #print(thresholded_preds_count)
        #print(outputs[0]['labels'])
        labels = [self.class_names[i-1] for i in outputs[0]['labels']]
        #print(labels)
        labels = labels[:thresholded_preds_count]

        # [1,1,1, 2, 2, 2, 3, 3]
        return masks, boxes, labels, scores

    def draw_segmentation_map(self, image, masks, boxes, labels):
        alpha = 1 
        beta = 0.6 # transparency for the segmentation map
        gamma = 0 # scalar added to each 
        segmentation_map = np.zeros((image.shape[0], image.shape[1]), dtype=np.uint8)
        result_masks = np.zeros((image.shape[0], image.shape[1]), dtype=np.uint8)

        for i in range(len(masks)):

            # TODO fix the color segmentation masks
            red_map = np.zeros_like(masks[i]).astype(np.uint8)
            green_map = np.zeros_like(masks[i]).astype(np.uint8)
            blue_map = np.zeros_like(masks[i]).astype(np.uint8)
            
            # apply a randon color mask to each object
            rect_color = (0,0,0)
            color = self.colors[random.randrange(0, len(self.colors))]
            red_map[masks[i] == 1], green_map[masks[i] == 1], blue_map[masks[i] == 1]  = self.class_to_colors[labels[i]]
            result_masks[masks[i] == 1] = 255
            # combine all the masks into a single image
            # change the format of mask to W,H, C

            # segmentation_map = np.stack([red_map, green_map, blue_map], axis=2)
            #convert the original PIL image into NumPy format
            image = np.array(image)
            # convert from RGB to OpenCV BGR format
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
            # apply mask on the image
            # cv2.addWeighted(image, alpha, segmentation_map, beta, gamma, image)
            # draw the bounding boxes around the objects
            cv2.rectangle(image, boxes[i][0], boxes[i][1], color=rect_color, 
                        thickness=2)
            # Get the centre coords of the rectangle-plaque-detection/src/visualizat
            x1 = boxes[i][0][0]
            y1 = boxes[i][0][1]
            x2 = boxes[i][1][0]
            y2 = boxes[i][1][1]
            x = int((x1 + x2) / 2)
            y = int((y1+y2) / 2)

            
            # put the label text above the objects
            cv2.putText(image , labels[i], (x1, y1-20), 
                        cv2.FONT_HERSHEY_SIMPLEX, 1, color, 
                        thickness=2, lineType=cv2.LINE_AA)
            
            # Convert Back
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        return image, result_masks

    def prepare_input(self, image):
    
        image_float_np = np.float32(image) / 255

        # define the torchvision image transforms
        transform = torchvision.transforms.Compose([
            torchvision.transforms.ToTensor(),
        ])

        input_tensor = transform(image)
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        input_tensor = input_tensor.to(device)
        # Add a batch dimension:
        input_tensor = input_tensor.unsqueeze(0)

        return input_tensor, image_float_np
    
    def predict(self, input_tensor, model, device, detection_threshold):
        outputs = model(input_tensor)
        # i- 1 zero indexing - the model outputs a non zero indexing format ( 1, 2, 3)
        pred_classes = [self.class_names[i-1] for i in outputs[0]['labels'].cpu().numpy()]
        pred_labels = outputs[0]['labels'].cpu().numpy()
        pred_scores = outputs[0]['scores'].detach().cpu().numpy()
        pred_bboxes = outputs[0]['boxes'].detach().cpu().numpy()        
        boxes, classes, labels, indices = [], [], [], []
        for index in range(len(pred_scores)):
            if pred_scores[index] >= detection_threshold:
                boxes.append(pred_bboxes[index].astype(np.int32))
                classes.append(pred_classes[index])
                labels.append(pred_labels[index])
                indices.append(index)
        boxes = np.int32(boxes)
        return boxes, classes, labels, indices

    def draw_boxes(self, boxes, labels, classes, image):
        for i, box in enumerate(boxes):
            color = self.colors[labels[i]]
            cv2.rectangle(
                image,
                (int(box[0]), int(box[1])),
                (int(box[2]), int(box[3])),
                color, 2
            )
            cv2.putText(image, classes[i], (int(box[0]), int(box[1] + 30)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 1,
                        lineType=cv2.LINE_AA)
        return image

    def make_result_dirs(self, folder_name):
        #folder_name = self.wandb.name + "_" + folder_name
        save_path = os.path.join(self.result_save_dir, self.wandb.name, folder_name)
        print("save_path",save_path)
        results_path = os.path.join(save_path, "results")
        if not os.path.exists(results_path):
            print("making results directory")
            os.makedirs(results_path)
        
        detections_path = os.path.join(save_path, "detections")
        if not os.path.exists(detections_path):
            print("making detections directory")
            os.makedirs(detections_path)
        
        masks_path = os.path.join(save_path, "masks")
        if not os.path.exists(masks_path):
            print("making masks directory")
            os.makedirs(masks_path)
        
        ablations_path = os.path.join(save_path, "ablations")
        if not os.path.exists(ablations_path):
            os.makedirs(ablations_path)
        
        pixel_count_path = os.path.join(self.result_save_dir, "pixel_count")
        if not os.path.exists(pixel_count_path):
            os.makedirs(pixel_count_path)
        
        csv_name = folder_name + "_quantify.csv"
        quantify_path = os.path.join(save_path, csv_name)

        self.results_path = results_path
        self.masks_path = masks_path
        self.detections_path = detections_path
        self.ablations_path = ablations_path
        self.quantify_path = quantify_path
        self.pixel_count_path = pixel_count_path

    def quantify_plaques(self, df, wandb_result, img_name, result_img, result_masks, boxes, labels, scores, total_brown_pixels):
        '''This function will take masks image and generate attributes like plaque
        count, area, eccentricity'''

        csv_result = []
        
    
        for i in range(len(labels)):

            props = {}
            data = {}
            # Here x and y axis are flipped
            total_true_LB = 0
            total_pre_LB = 0
            total_false_LB = 0

            if len(boxes)!= 0:

                x1 = boxes[i][0][1]
                x2 =  boxes[i][1][1]
                y1 = boxes[i][0][0]
                y2 = boxes[i][1][0]

               
                cropped_img = result_img[x1:x2, y1:y2]
                cropped_img_mask = result_masks[x1:x2, y1:y2]

                ret, bw_img = cv2.threshold(cropped_img_mask,0,255,cv2.THRESH_BINARY)

                kernel = np.ones((5,5),np.uint8)
                
                # Closing operation Dilation followed by erosion
                closing = cv2.morphologyEx(bw_img, cv2.MORPH_CLOSE, kernel)
                regions = regionprops(closing)

                for props in regions:

                    if labels[i] == "True":
                        total_true_LB+=1
                    elif labels[i] == "Pre":
                        total_pre_LB+=1
                    elif labels[i] == "False":
                        total_false_LB+=1
                    
                    data_record = pd.DataFrame.from_records([{ 'image_name': img_name, 'label': labels[i] , 'confidence': scores[i],
                                                               'brown_pixels': total_brown_pixels,
                                                               'True': total_true_LB, 'Pre': total_pre_LB, 'False': total_false_LB,
                                                               'centroid': props.centroid, 'eccentricity': props.eccentricity, 
                                                               'area': props.area, 'equivalent_diameter': props.equivalent_diameter}])
                    #wandb_result.append([img_name, wandb.Image(cropped_img), wandb.Image(cropped_img_mask), labels[i], scores[i], 
                    #                     total_brown_pixels, props.centroid, props.eccentricity, props.area, props.equivalent_diameter])

                    df = pd.concat([df, data_record], ignore_index=True)
                   
        
        return df
 
    def generate_results(self):
        # This will help us create a different color for each class
        # Load Trained 
        
        self.model.load_state_dict(torch.load(self.model_input_path))
    
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.eval().to(device)

        test_folders = glob.glob(os.path.join(self.test_input_path, "*"))
        print(test_folders)
        # Test images from each WSI folder
        test_folders = sorted(test_folders)

        final_df = pd.DataFrame()
        
        for test_folder in tqdm(test_folders):
            print("Running test fold \n", test_folder)
            #test_folder = os.path.join(test_folder,"images")
            #folder_name = os.path.basename(test_folder)
            folder_name = test_folder.split("/")[-1]
            print(folder_name)
            if folder_name == "labels":
                continue
            
            # make all necessary folders
            self.make_result_dirs(folder_name)
            test_folder = os.path.join(test_folder,"images")
            images = glob.glob(os.path.join(test_folder, '*.png'))
            print(len(images))
            i = 0
            df = pd.DataFrame()
            wandb_result = []
            total_brown_pixels = 0
            total_image_pixels = 0

            for img in tqdm(images):
                result_img = 0
                img_name = os.path.basename(img).split('.')[0]
        
                image = np.array(Image.open(img))

                total_image_pixels+= image.shape[0] * image.shape[1]
                # Check if image has alpha channel
                if image.shape[2] == 4:
                    image = image[:,:, :3]

                input_tensor, image_float_np = self.prepare_input(image)
                masks, boxes, labels, scores = self.get_outputs(input_tensor, self.model, self.detection_threshold)
                
                #img1 = img.replace("train","Unnormalized")
                #image1 = np.array(Image.open(img1))
                result_img, result_masks = self.draw_segmentation_map(image, masks, boxes, labels)

                total_brown_pixels+= self.get_brown_pixel_cnt(image, img_name)

                df = self.quantify_plaques(df, wandb_result, img_name, result_img, result_masks, boxes, labels, scores, total_brown_pixels)
            if len(final_df)>0:
                final_df = pd.concat([final_df,df], ignore_index=True)
            else:
                final_df= df
            final_df["test_folder"] = folder_name
        final_df["label"]=True
        predicted_labels_merged = pd.merge(self.ground_truth, final_df, on="image_name", how="left")
        predicted_labels_merged["ground_truth"]=True
        #predicted_labels_merged = pd.merge(final_df,self.ground_truth,on="image_name", how="left")
        predicted_labels_merged["matched"]=np.where(predicted_labels_merged["label"]==predicted_labels_merged["ground_truth"],1,0)
        #predicted_labels_merged["wsi_id"]=predicted_labels_merged["image_name"].apply(lambda l:l.split("_CG_")[0])
        #self.ground_truth["wsi_id"]=self.ground_truth["image_name"].apply(lambda l:l.split("_CG_")[0])
        #self.ground_truth["wsi_to_keep"] = self.ground_truth["wsi_id"].isin(predicted_labels_merged["wsi_id"].unique())
        #ground_truth_1=self.ground_truth[self.ground_truth["wsi_to_keep"]==True]
        #ground_truth_1 = pd.merge(ground_truth_1, final_df,on="image_name", how="left")
        #predicted_labels_merged["LBD_type"] = np.where(ground_truth_1["image_name"].str.startswith("PD"),"PDD","DLB")
        predicted_labels_merged["LBD_type"] = np.where(predicted_labels_merged["image_name"].str.startswith("PD"),"PDD","DLB")
        #ground_truth_1["matched"]=np.where(ground_truth_1["label"].isna(),0,1)
        t1 = predicted_labels_merged.groupby(["LBD_type","wsi_id"]).agg({"image_name":"count","matched":"sum"}).reset_index()
        t1.columns = ["LBD_type","wsi_id","gt_count","pred_matched_count"]
        #t2 = ground_truth_1.groupby(["LBD_type","wsi_id"]).agg({"image_name":"count","matched":"sum"}).reset_index()
        #t2.columns = ["LBD_type","wsi_id","gt_count","gt_matched_count"]
        #gt_pd = pd.merge(t1,t2,on=["LBD_type","wsi_id"],how="left")
        print(predicted_labels_merged)
        return final_df, t1
           

In [12]:
model_input_path.split("/")[-1]

'trim-puddle-222_mrcnn_model_50.pth'

In [13]:
explain = ExplainPredictions(model, model_input_path = model_input_path, test_input_path=input_path, 
                                    detection_threshold=0.75, wandb='', save_result=True, ablation_cam=True, save_thresholds=False, ground_truth = ground_truth)
explain.generate_results()

[[11.57688727 80.50615604]
 [75.57681242 40.9162297 ]]
['/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Train_val_LB/val/PD306_Syn1_CG.svs', '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Train_val_LB/val/PD013_Syn1_CG.svs', '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Train_val_LB/val/PD334_Syn1_CG.svs', '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Train_val_LB/val/PD295_Syn1_CG.svs', '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Train_val_LB/val/11_063_CG_aSyn_x200.svs', '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Train_val_LB/val/02_021_Syn1_CG_200x.svs', '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Train_val_LB/val/PD311_Syn1_CG.svs', '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Train_val_LB/val/01_104_Syn1_CG_200x.svs', '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Train_val_LB/val/PD309_Syn1_CG.svs', '/gladstone/finkbeiner/steve/work/data/npsad

  0%|          | 0/18 [00:00<?, ?it/s]

Running test fold 
 /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Train_val_LB/val/00_1027_Syn1_CG_200x.svs
00_1027_Syn1_CG_200x.svs
12


  6%|▌         | 1/18 [00:03<01:03,  3.76s/it]

Running test fold 
 /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Train_val_LB/val/00_1108_Syn1_CG_200x.svs
00_1108_Syn1_CG_200x.svs
21


...

Running test fold 
 /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Train_val_LB/val/02_019_Syn1_CG_200x.svs
02_019_Syn1_CG_200x.svs
4


 28%|██▊       | 5/18 [00:25<01:02,  4.77s/it]

Running test fold 
 /gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Train_val_LB/val/02_021_Syn1_CG_200x.svs
02_021_Syn1_CG_200x.svs
78


 28%|██▊       | 5/18 [00:26<01:08,  5.24s/it]


KeyboardInterrupt: 

In [26]:
predicted_labels_merged = pd.merge(ground_truth, output_all_thresh, on="image_name", how="left")

In [30]:
predicted_labels_merged["matched"]=np.where(predicted_labels_merged["label"]==predicted_labels_merged["ground_truth"],1,0)

In [29]:
predicted_labels_merged["label"].isna().sum()

94

In [ ]:
output_all_thresh = pd.DataFrame()
matched_result =  pd.DataFrame()
#for th in [0.5,0.75]:
for th in [0.10, 0.15, 0.20, 0.25,0.30, 0.35, 0.40, 0.45,0.5, 0.55,0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]:
    explain = ExplainPredictions(model, model_input_path = model_input_path, test_input_path=input_path, 
                                    detection_threshold=th, wandb=' ', save_result=False, ablation_cam=False, save_thresholds=False,ground_truth=ground_truth)
    df, gt_pd = explain.generate_results()
    gt_pd["threshold"]=th
    df["threshold"] = th
    if len(output_all_thresh)>0:
        output_all_thresh = pd.concat([output_all_thresh,df], ignore_index=True)
        matched_result = pd.concat([matched_result,gt_pd], ignore_index=True)
    else:
        output_all_thresh= df
        matched_result=gt_pd

In [59]:
len(ground_truth)

515

In [61]:
d1 = output_all_thresh[output_all_thresh["threshold"]==0.75]

In [62]:
d2 = pd.merge(ground_truth,d1,on="image_name",how="left")
1-(d2["label"].isna().sum()/len(d2))

0.9134860050890585

In [39]:
matched_result_agg = matched_result.groupby(["LBD_type","threshold"]).agg({"pred_matched_count":"sum","gt_count":"sum"}).reset_index()

In [40]:
matched_result_agg["recall_eq"]=matched_result_agg["pred_matched_count"]/matched_result_agg["gt_count"]
#matched_result_agg["recall_eq"]=matched_result_agg["gt_matched_count"]/matched_result_agg["gt_count"]

In [41]:
matched_result_agg

,LBD_type,threshold,pred_matched_count,gt_count,recall_eq
0,DLB,0.10,751,796,0.943467
1,DLB,0.15,703,748,0.939840
2,DLB,0.20,667,712,0.936798
3,DLB,0.25,644,689,0.934688
4,DLB,0.30,627,672,0.933036
5,DLB,0.35,603,648,0.930556
6,DLB,0.40,576,622,0.926045
7,DLB,0.45,548,594,0.922559
8,DLB,0.50,535,581,0.920826
9,DLB,0.55,521,568,0.917254


In [53]:
(535+802)/(850+581)

0.9343116701607268

In [42]:
matched_result_agg.to_csv("/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Reports/figures/royal-butterfly-145_mrcnn_model_10/agg_recall_at_var_thres.csv")

In [43]:
output_all_thresh.to_csv("/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Reports/figures/royal-butterfly-145_mrcnn_model_10/image_label_at_var_thres.csv")

## Plots

In [1]:
import plotly.express as px
import pandas as pd

In [10]:
matched_result_agg = pd.read_csv("/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Reports/figures/royal-butterfly-145_mrcnn_model_10/agg_recall_at_var_thres1.csv")

In [11]:
tmp = matched_result_agg[matched_result_agg["threshold"]>=0.5]

In [17]:
tmp["recall_eq"] =tmp["recall_eq"]*100

/tmp/ipykernel_478161/3338992577.py:1: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [22]:
fig = px.line(tmp, x="threshold", y="recall_eq", color='LBD_type', title='Recall_eq',symbol="LBD_type", 
              labels=dict(recall_eq="% match"))
fig.update_layout({"plot_bgcolor": "rgba(0, 0, 0, 0)"})
fig.update_xaxes(showline=True, linewidth=1, linecolor='black',minor_ticks="inside")
fig.update_yaxes(showline=True, linewidth=1, linecolor='black',minor_ticks="inside")

fig.show()